In [1]:
import os
import numpy as np
import pandas as pd
import librosa

# Dataset paths
DATA_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

TRAIN_AUDIO_DIR = os.path.join(DATA_PATH, "train")
TEST_AUDIO_DIR = os.path.join(DATA_PATH, "test")

TRAIN_CSV = os.path.join(DATA_PATH, "train.csv")
TEST_CSV = os.path.join(DATA_PATH, "test.csv")

# Load metadata
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

Train shape: (769, 2)
Test shape: (216, 2)
Train columns: ['filename', 'label']
Test columns: ['filename', 'label']


In [2]:
def extract_advanced_features(filename, audio_dir):
    path = os.path.join(audio_dir, filename)
    y, sr = librosa.load(path, sr=16000, mono=True)

    features = {}

    # 1. Duration
    duration = len(y) / sr
    features["duration"] = duration

    # 2. RMS Energy
    rms = librosa.feature.rms(
        y=y,
        frame_length=2048,
        hop_length=512
    )[0]

    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["rms_min"] = np.min(rms)
    features["rms_max"] = np.max(rms)

    for q in [10, 25, 50, 75, 90]:
        features[f"rms_q{q}"] = np.percentile(rms, q)

    features["rms_range"] = np.max(rms) - np.min(rms)

    # 3. Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=2048,
        hop_length=512
    )[0]

    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    for q in [10, 25, 50, 75, 90]:
        features[f"zcr_q{q}"] = np.percentile(zcr, q)

    # 4. Spectral features
    centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr, hop_length=512
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y, sr=sr, hop_length=512
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=y, sr=sr, hop_length=512
    )[0]

    flatness = librosa.feature.spectral_flatness(
        y=y, hop_length=512
    )[0]

    spectral_features = {
        "centroid": centroid,
        "bandwidth": bandwidth,
        "rolloff": rolloff,
        "flatness": flatness
    }

    for name, values in spectral_features.items():
        features[f"{name}_mean"] = np.mean(values)
        features[f"{name}_std"] = np.std(values)
        features[f"{name}_min"] = np.min(values)
        features[f"{name}_max"] = np.max(values)

        for q in [25, 50, 75]:
            features[f"{name}_q{q}"] = np.percentile(values, q)

    # 5. Spectral Contrast
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr,
        hop_length=512
    )

    for i in range(contrast.shape[0]):
        features[f"contrast_{i+1}_mean"] = np.mean(contrast[i])
        features[f"contrast_{i+1}_std"] = np.std(contrast[i])
        features[f"contrast_{i+1}_median"] = np.median(contrast[i])

    # 6. MFCC + Delta + Delta-Delta
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13,
        hop_length=512
    )

    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

        features[f"mfcc_delta_{i+1}_mean"] = np.mean(mfcc_delta[i])
        features[f"mfcc_delta_{i+1}_std"] = np.std(mfcc_delta[i])

        features[f"mfcc_delta2_{i+1}_mean"] = np.mean(mfcc_delta2[i])
        features[f"mfcc_delta2_{i+1}_std"] = np.std(mfcc_delta2[i])

    # 7. Mel-spectrogram statistics
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=20,
        hop_length=512
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    for i in range(20):
        features[f"mel_{i+1}_mean"] = np.mean(mel_db[i])
        features[f"mel_{i+1}_std"] = np.std(mel_db[i])

    # 8. Onset / temporal activity
    onset_strength = librosa.onset.onset_strength(
        y=y,
        sr=sr,
        hop_length=512
    )

    features["onset_mean"] = np.mean(onset_strength)
    features["onset_std"] = np.std(onset_strength)
    features["onset_max"] = np.max(onset_strength)

    for q in [25, 50, 75]:
        features[f"onset_q{q}"] = np.percentile(onset_strength, q)

    # 9. Tempo
    tempo = librosa.feature.tempo(
        onset_envelope=onset_strength,
        sr=sr,
        hop_length=512
    )[0]

    features["tempo"] = tempo

    # 10. Pitch
    try:
        f0 = librosa.yin(
            y,
            fmin=70,
            fmax=400,
            sr=sr,
            frame_length=2048,
            hop_length=512
        )

        valid_f0 = f0[np.isfinite(f0)]

        if len(valid_f0) > 0:
            features["pitch_mean"] = np.mean(valid_f0)
            features["pitch_std"] = np.std(valid_f0)
            features["pitch_min"] = np.min(valid_f0)
            features["pitch_max"] = np.max(valid_f0)
            features["pitch_median"] = np.median(valid_f0)
            features["pitch_range"] = np.max(valid_f0) - np.min(valid_f0)
            features["pitch_q25"] = np.percentile(valid_f0, 25)
            features["pitch_q75"] = np.percentile(valid_f0, 75)
        else:
            for name in [
                "pitch_mean", "pitch_std", "pitch_min",
                "pitch_max", "pitch_median", "pitch_range",
                "pitch_q25", "pitch_q75"
            ]:
                features[name] = 0.0

    except Exception:
        for name in [
            "pitch_mean", "pitch_std", "pitch_min",
            "pitch_max", "pitch_median", "pitch_range",
            "pitch_q25", "pitch_q75"
        ]:
            features[name] = 0.0

    return features


print("Advanced feature extractor is ready.")

Advanced feature extractor is ready.


In [3]:
advanced_train_features = []

for i, filename in enumerate(train_df["filename"]):
    features = extract_advanced_features(filename, TRAIN_AUDIO_DIR)
    features["filename"] = filename
    advanced_train_features.append(features)

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(train_df)} files")

advanced_train = pd.DataFrame(advanced_train_features)

print("\nAdvanced training features shape:", advanced_train.shape)
print("Missing values:", advanced_train.isna().sum().sum())

Processed 50/769 files
Processed 100/769 files
Processed 150/769 files
Processed 200/769 files
Processed 250/769 files
Processed 300/769 files
Processed 350/769 files
Processed 400/769 files
Processed 450/769 files
Processed 500/769 files
Processed 550/769 files
Processed 600/769 files
Processed 650/769 files
Processed 700/769 files
Processed 750/769 files

Advanced training features shape: (769, 201)
Missing values: 0


In [4]:
# Merge labels with advanced features
advanced_analysis = advanced_train.merge(
    train_df[["filename", "label"]],
    on="filename",
    how="inner"
)

# Calculate correlation of every numeric feature with the target
feature_cols = [
    col for col in advanced_train.columns
    if col != "filename"
]

correlations = (
    advanced_analysis[feature_cols + ["label"]]
    .corr()["label"]
    .drop("label")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("Top 25 features by absolute correlation with grammar score:")
print(correlations.head(25))

Top 25 features by absolute correlation with grammar score:
mfcc_delta2_2_std    0.638508
mfcc_2_std           0.637825
mfcc_delta_2_std     0.635859
mfcc_3_std           0.623558
mfcc_delta2_7_std    0.613625
mfcc_delta2_9_std    0.612057
mfcc_delta2_4_std    0.611317
rolloff_min         -0.610588
mfcc_delta_3_std     0.610419
mfcc_delta2_6_std    0.607893
mfcc_delta_4_std     0.606549
mfcc_delta_7_std     0.606304
flatness_q50        -0.602032
flatness_q25        -0.601069
rms_min             -0.600602
rms_q10             -0.600169
centroid_min        -0.597992
mfcc_delta_9_std     0.597958
mfcc_delta_6_std     0.596026
mfcc_delta2_3_std    0.595812
mfcc_4_std           0.592260
mfcc_6_std           0.590394
mfcc_7_std           0.583290
mfcc_9_std           0.580482
mfcc_delta2_8_std    0.576019
Name: label, dtype: float64


In [5]:
# Select advanced numeric features
advanced_feature_cols = [
    col for col in advanced_train.columns
    if col != "filename"
]

X_advanced = advanced_train[advanced_feature_cols].copy()
y_advanced = train_df["label"].values

print("Advanced feature matrix:", X_advanced.shape)
print("Target shape:", y_advanced.shape)
print("Total features:", len(advanced_feature_cols))

Advanced feature matrix: (769, 200)
Target shape: (769,)
Total features: 200


In [6]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_predictions_advanced = np.zeros(len(y_advanced))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_advanced), 1):

    X_train = X_advanced.iloc[train_idx]
    X_val = X_advanced.iloc[val_idx]

    y_train = y_advanced[train_idx]
    y_val = y_advanced[val_idx]

    model = ExtraTreesRegressor(
        n_estimators=500,
        max_features=0.8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    cv_predictions_advanced[val_idx] = model.predict(X_val)

    fold_rmse = np.sqrt(
        mean_squared_error(y_val, cv_predictions_advanced[val_idx])
    )

    print(f"Fold {fold} RMSE: {fold_rmse:.4f}")

overall_rmse = np.sqrt(
    mean_squared_error(y_advanced, cv_predictions_advanced)
)

overall_pearson = pearsonr(
    y_advanced,
    cv_predictions_advanced
)[0]

print("\nOverall CV RMSE:", round(overall_rmse, 4))
print("Overall CV Pearson:", round(overall_pearson, 4))

Fold 1 RMSE: 0.6859
Fold 2 RMSE: 0.7389
Fold 3 RMSE: 0.7953
Fold 4 RMSE: 0.7168
Fold 5 RMSE: 0.7967

Overall CV RMSE: 0.7479
Overall CV Pearson: 0.7992


In [7]:
from sklearn.feature_selection import SelectKBest, f_regression

feature_counts = [10, 20, 30, 40, 60]

selected_results = []

for k in feature_counts:
    selector = SelectKBest(
        score_func=f_regression,
        k=k
    )

    X_selected = selector.fit_transform(X_advanced, y_advanced)

    selected_model = ExtraTreesRegressor(
        n_estimators=500,
        max_features=0.8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    selected_cv_predictions = np.zeros(len(y_advanced))

    for train_idx, val_idx in kf.split(X_selected):

        selected_model.fit(
            X_selected[train_idx],
            y_advanced[train_idx]
        )

        selected_cv_predictions[val_idx] = selected_model.predict(
            X_selected[val_idx]
        )

    rmse = np.sqrt(
        mean_squared_error(
            y_advanced,
            selected_cv_predictions
        )
    )

    pearson = pearsonr(
        y_advanced,
        selected_cv_predictions
    )[0]

    selected_results.append({
        "features": k,
        "rmse": rmse,
        "pearson": pearson
    })

    print(
        f"Top {k:2d} features -> "
        f"RMSE: {rmse:.4f}, "
        f"Pearson: {pearson:.4f}"
    )

selected_results_df = pd.DataFrame(selected_results)

print("\nResults:")
print(selected_results_df)

Top 10 features -> RMSE: 0.8360, Pearson: 0.7377
Top 20 features -> RMSE: 0.8160, Pearson: 0.7522
Top 30 features -> RMSE: 0.8104, Pearson: 0.7563
Top 40 features -> RMSE: 0.8071, Pearson: 0.7588
Top 60 features -> RMSE: 0.7929, Pearson: 0.7691

Results:
   features      rmse   pearson
0        10  0.836031  0.737653
1        20  0.815968  0.752236
2        30  0.810407  0.756320
3        40  0.807066  0.758822
4        60  0.792882  0.769052


In [8]:
# Compare the existing CV predictions from Experiment 01
# with the advanced 200-feature predictions

print("Experiment 02 RMSE:", 
      np.sqrt(mean_squared_error(y_advanced, cv_predictions_advanced)))

print("Experiment 02 Pearson:",
      pearsonr(y_advanced, cv_predictions_advanced)[0])

print("\nPrediction statistics:")
print("Mean:", cv_predictions_advanced.mean())
print("Std :", cv_predictions_advanced.std())
print("Min :", cv_predictions_advanced.min())
print("Max :", cv_predictions_advanced.max())

Experiment 02 RMSE: 0.7479159507574938
Experiment 02 Pearson: 0.7992228661979558

Prediction statistics:
Mean: 3.30927915041179
Std : 0.9152188973887428
Min : 0.0
Max : 4.850666666666668


In [9]:
# Recreate Experiment 01 features in this notebook
# using the same 37 original acoustic features.

def extract_original_features(filename, audio_dir):
    path = os.path.join(audio_dir, filename)
    y, sr = librosa.load(path, sr=16000, mono=True)

    features = {}

    features["duration"] = len(y) / sr

    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features["centroid_mean"] = np.mean(centroid)
    features["centroid_std"] = np.std(centroid)

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features["bandwidth_mean"] = np.mean(bandwidth)
    features["bandwidth_std"] = np.std(bandwidth)

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features["rolloff_mean"] = np.mean(rolloff)
    features["rolloff_std"] = np.std(rolloff)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    return features


original_train_features = []

for i, filename in enumerate(train_df["filename"]):
    features = extract_original_features(
        filename,
        TRAIN_AUDIO_DIR
    )
    features["filename"] = filename
    original_train_features.append(features)

original_train = pd.DataFrame(original_train_features)

original_feature_cols = [
    col for col in original_train.columns
    if col != "filename"
]

X_original = original_train[original_feature_cols].copy()

print("Original feature matrix:", X_original.shape)
print("Missing values:", X_original.isna().sum().sum())

Original feature matrix: (769, 37)
Missing values: 0


In [10]:
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Prepare the best feature representations we already have
# ---------------------------------------------------------

# Experiment 01 combined 60 features
X_60 = pd.concat(
    [
        X_original.reset_index(drop=True),
        advanced_train[
            [c for c in advanced_train.columns if c != "filename"]
        ].iloc[:, :23].reset_index(drop=True)
    ],
    axis=1
)

# Make sure dimensions are correct
print("Original:", X_original.shape)
print("60-feature:", X_60.shape)
print("Target:", y_advanced.shape)

# ---------------------------------------------------------
# 5-fold OOF predictions
# ---------------------------------------------------------

kf_blend = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_et_60 = np.zeros(len(y_advanced))
oof_rf_60 = np.zeros(len(y_advanced))
oof_hgb_60 = np.zeros(len(y_advanced))

for fold, (train_idx, val_idx) in enumerate(
    kf_blend.split(X_60), 1
):

    Xtr = X_60.iloc[train_idx]
    Xva = X_60.iloc[val_idx]

    ytr = y_advanced[train_idx]
    yva = y_advanced[val_idx]

    # Model 1: Extra Trees
    et = ExtraTreesRegressor(
        n_estimators=800,
        max_features=0.8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    et.fit(Xtr, ytr)
    oof_et_60[val_idx] = et.predict(Xva)

    # Model 2: Random Forest
    rf = RandomForestRegressor(
        n_estimators=700,
        max_features=0.8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(Xtr, ytr)
    oof_rf_60[val_idx] = rf.predict(Xva)

    # Model 3: Histogram Gradient Boosting
    hgb = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.04,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    )

    hgb.fit(Xtr, ytr)
    oof_hgb_60[val_idx] = hgb.predict(Xva)

    print(f"Fold {fold} completed")

# ---------------------------------------------------------
# Evaluate individual models
# ---------------------------------------------------------

predictions = {
    "ExtraTrees": oof_et_60,
    "RandomForest": oof_rf_60,
    "HistGradientBoosting": oof_hgb_60
}

for name, pred in predictions.items():

    rmse = np.sqrt(
        mean_squared_error(y_advanced, pred)
    )

    corr = pearsonr(
        y_advanced,
        pred
    )[0]

    print(
        f"{name}: "
        f"RMSE={rmse:.4f}, "
        f"Pearson={corr:.4f}"
    )

# ---------------------------------------------------------
# Test many blend weights
# ---------------------------------------------------------

best_rmse = float("inf")
best_weights = None
best_blend = None

for w1 in np.arange(0, 1.01, 0.05):

    for w2 in np.arange(0, 1.01 - w1, 0.05):

        w3 = 1.0 - w1 - w2

        blend = (
            w1 * oof_et_60
            + w2 * oof_rf_60
            + w3 * oof_hgb_60
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_advanced,
                blend
            )
        )

        if rmse < best_rmse:
            best_rmse = rmse
            best_weights = (w1, w2, w3)
            best_blend = blend.copy()

best_pearson = pearsonr(
    y_advanced,
    best_blend
)[0]

print("\nBEST BLEND")
print("Weights:")
print("ExtraTrees:", best_weights[0])
print("RandomForest:", best_weights[1])
print("HistGradientBoosting:", best_weights[2])
print("RMSE:", best_rmse)
print("Pearson:", best_pearson)


Original: (769, 37)
60-feature: (769, 60)
Target: (769,)
Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed
ExtraTrees: RMSE=0.7619, Pearson=0.7904
RandomForest: RMSE=0.7863, Pearson=0.7738
HistGradientBoosting: RMSE=0.7898, Pearson=0.7705

BEST BLEND
Weights:
ExtraTrees: 0.8
RandomForest: 0.0
HistGradientBoosting: 0.19999999999999996
RMSE: 0.7602818449210264
Pearson: 0.7909206003517879
